# Load model weights to GPU

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-0.6B-Base"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model
model = AutoModelForCausalLM.from_pretrained(model_name)

# Example of generating text (optional)
inputs = tokenizer("Hello, world!", return_tensors="pt")
outputs = model.generate(inputs["input_ids"], max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
print(inputs)
print(outputs)

In [ ]:
del model

# Send request to LLM API

In [ ]:
from openai import OpenAI

import os
API_KEY = os.getenv("OPENAI_API_KEY")
# Initialize the client to connect to OpenAI API service
client = OpenAI(
    api_key=API_KEY
)

# Request a chat completion with streaming enabled
response = client.chat.completions.create(
    model='gpt-4o', # Specify the model name
    messages=[
        {'role': 'user', 'content': 'Hello world.'}
    ], stream = False,
)

# Process and print the streamed response
print(response.choices[0].message.content)

# Host LLM models with Ollama

In [ ]:
# Model selection
MODEL_NAME = "llama3:8b" # TODO: For now, use "llama3:8b"
%env OLLAMA_CONTEXT_LENGTH=8000
%env OLLAMA_HOST=0.0.0.0
%env OLLAMA_KEEP_ALIVE=-1

In [ ]:
!apt-get install -y lshw pciutils
!nvcc --version
!nvidia-smi

# Print available memory
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print(f"\nAvailable RAM: {ram_gb:.1f} GB")

In [ ]:
!apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time
import requests
import threading

OLLAMA_BASE_URL = "http://localhost:11434"

# Start Ollama server in a background thread.
def start_ollama():
    subprocess.call(["ollama", "serve"])

ollama_thread = threading.Thread(target=start_ollama, daemon=True)
ollama_thread.start()

# Wait for Ollama HTTP API to be ready before pulling the model.
def wait_for_ollama(timeout=60):
    for i in range(timeout):
        try:
            r = requests.get(OLLAMA_BASE_URL)
            if r.status_code in [200, 404]:
                print(f"Ollama is up after {i + 1}s.")
                return
        except requests.exceptions.ConnectionError:
            pass
        print(f"Waiting for Ollama to start... {i + 1}s")
        time.sleep(1)
    raise RuntimeError("Ollama did not start in time.")

wait_for_ollama()

# Pull model only after the server is ready.
!ollama pull {MODEL_NAME}
!ollama list

# Sending requests

### Streaming Enable

In [ ]:
from openai import OpenAI

# Initialize the client to connect to the local Ollama instance
client = OpenAI(
    base_url=f"{OLLAMA_BASE_URL}/v1/",
    api_key='ollama', # The api_key is required but ignored by Ollama
)

# Request a chat completion with streaming enabled
response_stream = client.chat.completions.create(
    model=MODEL_NAME, # Specify the model name
    messages=[
        {'role': 'user', 'content': 'Explain the concept of quantum physics in a simple way, like I am a beginner.'}
    ],
    stream=True, # Enable streaming
)

# Process and print the streamed response
print("AI Response (Streaming):")
for chunk in response_stream:
    # Extract the token and print it
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)

print()

### Streaming disable

In [ ]:
# Make a chat completion request with streaming disabled
response = client.chat.completions.create(
    model=MODEL_NAME, # Specify the model, e.g., llama3
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain what the difference is between a Llama 3 8B and 70B model in a short sentence."}
    ],
    stream=False,
)

# Print the complete response content
print(response.choices[0].message.content)

# Expose as API

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import re

# Run cloudflared tunnel in background and get the public URL
cloudflared_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in cloudflared_proc.stdout:
    print(line.strip())
    match = re.search(r'(https://.*\.trycloudflare\.com)', line)
    if match:
        public_url = match.group(1)
        break

if public_url:
    print(f"\nPublic URL for Ollama:\n{public_url}")
else:
    raise RuntimeError("Could not find public Cloudflare URL.")

In [ ]:
client_new = OpenAI(
    base_url=f"{public_url}/v1",
    api_key='ollama', # The api_key is required but ignored by Ollama
)

response = client_new.chat.completions.create(
    model=MODEL_NAME, # Specify the model, e.g., llama3
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain what the difference is between a Llama 3 8B and 70B model in a short sentence."}
    ],
    stream=False, # Set stream to False to disable streaming
)

# Print the complete response content
print(response.choices[0].message.content)

usage = getattr(response, "usage", None)
if usage:
    print("Input tokens:", usage.prompt_tokens)
    print("Output tokens:", usage.completion_tokens)
    print("Total tokens:", usage.total_tokens)

# Experiment with temperature and top_p

In [ ]:
def test_llama3(client, prompt, temp, top_p):
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a creative writer."},
            {"role": "user", "content": prompt}
        ],
        temperature=temp,
        top_p=top_p,
    )
    return completion.choices[0].message.content

prompt = "Write a one-sentence story about a robot."

# 1. Deterministic/Conservative (Low Temp/Low Top_p)
print("--- Conservative ---")
print(test_llama3(client_new, prompt, 0.1, 0.1))

# 2. Creative/Random (High Temp/High Top_p)
print("\n--- Creative ---")
print(test_llama3(client_new, prompt, 1.5, 0.95))

# Chatbot loop

### Without context

In [ ]:
while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    response = client_new.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": user_input}
        ],
        max_tokens=300
    )

    print("Bot:", response.choices[0].message.content)

### With context

In [ ]:
messages = []

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    messages.append({"role": "user", "content": user_input})
    messages = messages[-6:]  # keep last 3 Q&A pairs (3 user + 3 assistant)

    response = client_new.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=300
    )

    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    print("Bot:", reply)